In [2]:
import torch 
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

In [3]:
# Set random seed for reproducibility
torch.random.manual_seed(42) 
# Create synthetic (empty) dataset with 20 samples (points) and 2 features (dis), then fill it with random values between -1 and 1
data = torch.empty(20, 2).uniform_(-1, 1)
labels = (data[:, 1] >= data[:, 0]).float().unsqueeze(1)  # Binary labels based on the first feature (dis)
dataset = TensorDataset(data, labels)

# Fixed subset of 4 samples for training
subset = torch.utils.data.Subset(dataset, [2, 3, 4, 5])
loader = DataLoader(subset, batch_size=4, shuffle=False)



In [4]:
# Define a simple linear model
model = nn.Linear(2, 1)  # Input: 2 features, Output: 1 (binary classification)
criterion = nn.BCEWithLogitsLoss()  # Binary Cross-Entropy Loss with logits 
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)  # Stochastic Gradient Descent optimizer


In [5]:
# Training loop
losses = []
for epoch in range(100):
    for x, y in loader:
        optimizer.zero_grad()  # Clear gradients
        loss = criterion(model(x), y)  # Compute loss
        loss.backward()  # Backpropagation
        optimizer.step()  # Update model parameters
    losses.append(loss.item())

print(f"Final Loss: {loss.item():.4f}")



Final Loss: 0.2390


In [6]:
# testset: 40 points evenly spread over the [-1, 1]^2 plane (8x5 grid)
test_data = torch.cartesian_prod(torch.linspace(-1, 1, 20), torch.linspace(-1, 1, 20))
test_labels = (test_data[:, 1] >= test_data[:, 0]).float().unsqueeze(1)  # Binary labels based on the first feature (dis)
test_dataset = TensorDataset(test_data, test_labels)
test_loader = DataLoader(test_dataset, batch_size=400, shuffle=False)



In [7]:
# Evaluate the model on the test set
model.eval()  # Set model to evaluation mode
with torch.no_grad():  # No need to compute gradients during evaluation
    for x, y in test_loader:
        logits = model(x)  # Get model predictions (logits)
        preds = (logits >= 0).float()  # Convert logits to binary predictions (threshold at 0)
        accuracy = (preds == y).float().mean()  # Calculate accuracy
print(f"Test Accuracy: {accuracy.item():.4f}")


Test Accuracy: 0.8550


In [8]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

correct = (preds == y).squeeze().numpy()  # Convert to 1D tensor for easier indexing
x_np = x.numpy()  # Convert to NumPy array for plotting

# Plot training loss
fig, (ax_loss, ax_scatter) = plt.subplots(1, 2, figsize=(12, 5))

ax_loss.plot(losses)    
ax_loss.set_xlabel('Epoch')
ax_loss.set_ylabel('Loss')
ax_loss.set_title('Training Loss')

# Scatter plot of test points colored by correctness
ax_scatter.scatter(x_np[correct, 0],  x_np[correct, 1],  c="green", marker="o", label="Correct")
ax_scatter.scatter(x_np[~correct, 0], x_np[~correct, 1], c="red",   marker="x", label="Incorrect")
ax_scatter.plot([-1, 1], [-1, 1], color='gray', linestyle='--', label='Decision Boundary (x0 = x1)')
ax_scatter.set_xlabel('x0')
ax_scatter.set_ylabel('x1')
ax_scatter.set_title('Test set classification results')
ax_scatter.legend()

plt.tight_layout()
#plt.show()
plt.savefig("result.png")
print("diagram saved as result.png")

diagram saved as result.png
